In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [346]:
import tensorflow as tf
from PIL import Image
import numpy as np
tf.get_logger().setLevel('ERROR')
from typing import Any

In [3]:
tf.keras.backend.clear_session()

In [4]:
from mobilenetv2ssd.core.config import load_config

In [5]:
experiment_path = "../configs/experiments/exp001_baseline.yaml"

In [6]:
config = load_config(experiment_path)

In [7]:
from mobilenetv2ssd.models.ssd.orchestration.targets_orch import building_training_targets
from mobilenetv2ssd.models.ssd.orchestration.conf_loss_orch import build_conf_loss
from mobilenetv2ssd.models.ssd.orchestration.hard_neg_orch import select_hard_negatives
from mobilenetv2ssd.models.ssd.orchestration.loss_orch import calculate_final_loss
from mobilenetv2ssd.models.factory import build_ssd_model
from mobilenetv2ssd.models.ssd.orchestration.post_process_orch import build_decoded_boxes

In [8]:
from mobilenetv2ssd.core.precision_config import PrecisionConfig
from mobilenetv2ssd.core.logger import build_logger_from_config, Logger

In [9]:
from datasets.voc import VOCDataset
from datasets.transforms import build_train_transforms, build_validation_transforms
from datasets.collate import create_training_dataset, create_validation_dataset

In [10]:
from training.optimizer import OptimizerFactory
from training.schedule import LearningRateSchedulerFactory
from training.amp import build_amp, AMPContext
from training.checkpoints import build_checkpoint_manager, CheckpointManager
from training.ema import build_ema, EMA
from training.metrics import build_metrics_from_config, convert_batch_images_to_metric_format,MetricsCollection

In [11]:
from mobilenetv2ssd.models.ssd.orchestration.priors_orch import build_priors_from_config

In [12]:
priors, priors_meta = build_priors_from_config(config)

I0000 00:00:1770671703.101801    3758 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770671703.208552    3758 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770671703.208622    3758 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770671703.210820    3758 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1770671703.210910    3758 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [13]:
priors_meta

{'image_size': (300, 300),
 'feature_map_sizes': [(38, 38), (19, 19), (10, 10), (5, 5), (3, 3), (2, 2)],
 'strides': [8, 16, 32, 64, 128, 224],
 'scales_per_layer': [[0.2, 0.264575131106459],
  [0.35, 0.4183300132670378],
  [0.5, 0.5700877125495689],
  [0.6499999999999999, 0.7211102550927978],
  [0.8, 0.8717797887081347],
  [0.95, 0.9746794344808963]],
 'ratios_per_layer': [[1, 2.0, 0.5],
  [1, 2.0, 3.0, 0.5, 0.3333],
  [1, 2.0, 3.0, 0.5, 0.3333],
  [1, 2.0, 0.5],
  [1, 2.0, 0.5],
  [1, 2.0, 0.5]],
 'number_of_anchors_per_layer': <tf.Tensor: shape=(6,), dtype=int32, numpy=array([8664, 3610, 1000,  150,   54,   24], dtype=int32)>,
 'cells_per_layer': <tf.Tensor: shape=(6,), dtype=int32, numpy=array([1444,  361,  100,   25,    9,    4], dtype=int32)>,
 'anchors_per_cell': <tf.Tensor: shape=(6,), dtype=int32, numpy=array([ 6, 10, 10,  6,  6,  6], dtype=int32)>,
 'total_number_of_anchors': <tf.Tensor: shape=(), dtype=int32, numpy=13502>,
 'fingerprint': '3a66af61ef40c7f3ef5b7099864d5e75'}

In [14]:
learning_schedule = LearningRateSchedulerFactory.build(config)

In [15]:
optimizer = OptimizerFactory.build(config, learning_schedule)

In [16]:
backbone_optimizer = OptimizerFactory.build(config, learning_schedule)

In [17]:
amp = build_amp(config, optimizer)

In [18]:
precision_config = amp.make_precision_config()

In [19]:
optimizer = amp.wrap_optimizer()

In [20]:
optimizer

In [21]:
metrics_manager = build_metrics_from_config(config)

In [22]:
logger = build_logger_from_config(config)

2026-02-09 16:15:06 | ℹ️ INFO       | Starting TensorBoard on port 6006...
2026-02-09 16:15:06 | ℹ️ INFO       | Log directory: /mnt/d/dev/MobileNetV2-SSD/runs/exp001_no_fingerprint/logs/20260209_161506/tensorboard
2026-02-09 16:15:09 | ℹ️ INFO       | TensorBoard is running!
2026-02-09 16:15:09 | ℹ️ INFO       | ------------------------------------------------------------
2026-02-09 16:15:09 | ℹ️ INFO       | Access URLs:
2026-02-09 16:15:09 | ℹ️ INFO       |   Local:        http://localhost:6006
2026-02-09 16:15:09 | ℹ️ INFO       |   Network:      http://127.0.1.1:6006
2026-02-09 16:15:09 | ℹ️ INFO       |   Remote/Cloud: http://YOUR_SERVER_IP:6006
2026-02-09 16:15:09 | ℹ️ INFO       | ------------------------------------------------------------
2026-02-09 16:15:09 | ℹ️ INFO       | For AWS/Cloud instances:
2026-02-09 16:15:09 | ℹ️ INFO       |   1. Ensure security group allows inbound traffic on port 6006
2026-02-09 16:15:09 | ℹ️ INFO       |   2. Use: http://<your-instance-public-

In [23]:
# Create the model
model = build_ssd_model(config,[6, 10, 10, 6, 6, 6])

[build_backbone] Loading existing weights from: /mnt/d/dev/MobileNetV2-SSD/src/mobilenetv2ssd/models/mobilenet_v2/weights/mobilenetv2_imagenet_notop_300x300_1.weights.h5


W0000 00:00:1770671713.136363    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.171485    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.175137    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.181527    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.194940    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.200844    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.203117    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.291336    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671713.296094    3758 gp

In [24]:
ema = build_ema(config, model)

In [25]:
ema._update_every

1

In [26]:
checkpoint_manager = build_checkpoint_manager(config, model, optimizer = optimizer, ema = ema)

In [27]:
checkpoint_manager._optimizer

In [28]:
checkpoint_manager.build_optimizer(var_group = model.trainable_variables)

In [29]:
compose = build_train_transforms(config)

In [30]:
data = VOCDataset(root = config['data']['root'], split = "train", classes_file = config['data']['classes_file'], use_difficult = False)

In [31]:
# Create the dataset
dataset = create_training_dataset(dataset = data, config = config, transform = compose)

In [32]:
info = next(iter(dataset.take(1)))

In [33]:
data[1].orig_size

(327, 500)

In [34]:
data[1].boxes

array([[270.,   1., 378., 176.],
       [ 57.,   1., 164., 150.]], dtype=float32)

In [35]:
print(info['boxes'])

tf.Tensor(
[[[ 0.10600001  0.19683258  0.94200003  0.95022625]
  [ 0.316       0.09954751  0.578       0.37782806]
  [-1.         -1.         -1.         -1.        ]]

 [[ 0.54        0.0030581   0.756       0.53822625]
  [ 0.114       0.0030581   0.328       0.45871556]
  [-1.         -1.         -1.         -1.        ]]

 [[ 0.28958333  0.00735294  0.775       0.7242647 ]
  [ 0.34375     0.24264704  0.6625      0.867647  ]
  [ 0.75208336  0.00367647  1.          0.4117647 ]]], shape=(3, 3, 4), dtype=float32)


In [36]:
def ssd_get_prior_stats(positive_mask: tf.Tensor, negative_mask: tf.Tensor):
    positive_mask = tf.cast(positive_mask, tf.int32)
    negative_mask = tf.cast(negative_mask, tf.int32)

    # Per image values
    positive_prior_per_image = tf.reduce_sum(positive_mask, axis = 1)
    negative_prior_per_image = tf.reduce_sum(negative_mask, axis = 1)

    # Number of positive & negative anchors
    number_positive = tf.reduce_sum(positive_prior_per_image)
    number_negative = tf.reduce_sum(negative_prior_per_image)

    return{
        "num_pos": int(number_positive.numpy()),
        "pos_min": int(tf.reduce_min(positive_prior_per_image).numpy()),
        "pos_mean": float(tf.reduce_mean(tf.cast(positive_prior_per_image, tf.float32)).numpy()),
        "pos_max": int(tf.reduce_max(positive_prior_per_image).numpy()),
        "num_neg": int(number_negative.numpy()),
        "neg_pos_ratio": float((tf.cast(number_negative, tf.float32) / tf.maximum(tf.cast(number_positive, tf.float32), 1.0)).numpy()),
        "zero_pos_frac": float(tf.reduce_mean(tf.cast(positive_prior_per_image == 0, tf.float32)).numpy()), 
    }

In [37]:
def training_step(config: dict[str,Any],model: tf.keras.Model, priors_cxcywh: tf.Tensor, batch: dict[str, Any], precision_config: PrecisionConfig, logger: Logger):
    
    # First get the batch elements from the dataset
    image, boxes, labels, gt_mask = batch['image'], batch['boxes'], batch['labels'], batch['gt_mask']

    tf.debugging.assert_equal(tf.rank(image), tf.constant(4, dtype = tf.int32), message = f"The image has rank : {tf.rank(image)}, expected: 4")
    
    # Building the training targets
    localization_targets, classification_targets, positive_mask, negative_mask, ignore_mask, diagnostics = building_training_targets(config = config, priors_cxcywh = priors_cxcywh, gt_labels= labels, gt_boxes_xyxy= boxes, gt_valid_mask= gt_mask, precision_config= precision_config)
    
    tf.debugging.assert_equal(tf.shape(classification_targets)[:2], tf.shape(localization_targets)[:2], message = f"The shapes between the are different between classification targets:{tf.shape(classification_targets)[:2]}, but expected {tf.shape(localization_targets)[:2]}")
    
    predicted_offsets, predicted_logits = model(image, training = True)

    tf.debugging.assert_equal(tf.shape(predicted_logits)[:2], tf.shape(localization_targets)[:2], message = f"The shapes between the are different between localization logits :{tf.shape(predicted_logits)[:2]}, but expected {tf.shape(localization_targets)[:2]}")

    tf.debugging.assert_equal(tf.shape(localization_targets), tf.shape(predicted_offsets), message=f"The shapes between the are different between localization targets:{tf.shape(localization_targets)}, and predicted_offsets:{tf.shape(predicted_offsets)}")

    # Calculating the confidence loss
    conf_loss, candidate_negative_mask = build_conf_loss(config = config, predicted_logits = predicted_logits, classification_targets = classification_targets, pos_mask = positive_mask, neg_mask = negative_mask, ignore_mask = ignore_mask, precision_config = precision_config)
    
    # Perform Hard negative mining
    selected_negative_mask = select_hard_negatives(config = config, conf_loss = conf_loss, positive_mask = positive_mask, negative_mask = candidate_negative_mask)

    # Logging the value
    priors_stats = ssd_get_prior_stats(positive_mask, selected_negative_mask)
    
    # Calculate the loss
    loss_dict = calculate_final_loss(config = config, predicted_offsets = predicted_offsets, predicted_logits = predicted_logits, localization_targets = localization_targets, classification_targets = classification_targets, positive_mask = positive_mask, negative_mask = selected_negative_mask, precision_config = precision_config)
    
    return loss_dict, priors_stats # Checking if the boxes are correctly formatted

In [38]:
training_step(config, model, priors, info, precision_config, logger)

W0000 00:00:1770671733.416524    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.444742    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.464344    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.481185    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.501202    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.516568    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.533034    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.548757    3758 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770671733.557672    3758 gp

({'total_loss': <tf.Tensor: shape=(), dtype=float32, numpy=27.489918>,
  'loc_loss': <tf.Tensor: shape=(), dtype=float32, numpy=8.06805>,
  'cls_loss': <tf.Tensor: shape=(), dtype=float32, numpy=19.421867>,
  'num_pos': <tf.Tensor: shape=(), dtype=int32, numpy=518>,
  'raw_num_pos': <tf.Tensor: shape=(), dtype=int32, numpy=518>,
  'raw_num_negative': <tf.Tensor: shape=(), dtype=int32, numpy=1554>,
  'num_negative': <tf.Tensor: shape=(), dtype=int32, numpy=1554>},
 {'num_pos': 518,
  'pos_min': 126,
  'pos_mean': 172.6666717529297,
  'pos_max': 243,
  'num_neg': 1554,
  'neg_pos_ratio': 3.0,
  'zero_pos_frac': 0.0})

In [39]:
def train_one_epoch(config: dict[str, Any], epoch: int, model: tf.keras.Model, train_dataset: tf.data.Dataset, optimizer: tf.keras.optimizers.Optimizer, priors_cxcywh: tf.Tensor, precision_config: PrecisionConfig, ema : EMA, amp: AMPContext, logger: Logger, global_step_offset: int = 0, log_every: int = 5, max_steps: int|None = None, backbone_grad_scale: float = 0.1):
    # TODO: If num_pos is 0 the skip the loc loss and only do a safe cls loss OR skip the update completely
    # Running counter of the loss value
    loss_meter = tf.keras.metrics.Mean(name="loss")

    global_step = global_step_offset

    # Creating the set for backbone variables
    backbone_var_ids = {id(var) for var in model.backbone.trainable_variables}
    
    
    for step, batch in enumerate(train_dataset):
        # Use Gradient Tape to get the losses
        with tf.GradientTape() as tape:
            with amp.autocast():
                loss_dict, prior_stats = training_step(config = config, model = model, priors_cxcywh = priors_cxcywh, batch = batch, precision_config = precision_config, logger= logger)
                total_loss = loss_dict['total_loss']

                # Making sure the optimizer operation is guarded since it can be disabled
                if isinstance(optimizer, tf.keras.mixed_precision.LossScaleOptimizer):
                    scaled_loss = optimizer.scale_loss(total_loss)
                else:
                    scaled_loss = total_loss

            
        gradients = tape.gradient(scaled_loss, model.trainable_variables)

        # Isolating the backbone gradients
        grads_and_vars = []
        for grad, var in zip(gradients, model.trainable_variables):
            if grad is None:
                continue

            # checking if the id of the variable is in the backbone variable
            if id(var) in backbone_var_ids:
                # This is an backbone variable
                grad = grad * backbone_grad_scale

            # Adding it to the list
            grads_and_vars.append((grad,var))
            
        if not grads_and_vars:
            global_step += 1
            continue

        optimizer.apply_gradients(grads_and_vars)

        # Updating the EMA based on the predefined conditions
        ema.update(global_step)

        # Adding to the running counter
        loss_meter.update_state(total_loss)

        if step % log_every == 0:
            logger.metric(f"Epoch {epoch}, Step {step}, Loss {float(total_loss.numpy())}, Num Pos: {int(loss_dict['num_pos'].numpy())}")
            logger.log_scalar(tag= "train/loss", value= total_loss.numpy(), step= global_step)
            logger.metric(f"Number of Positive Priors: {prior_stats['num_pos']}")
            logger.metric(f"Number of Negative Priors: {prior_stats['num_neg']}")
            logger.metric(f"Number of Negative to Positive Ratio: {prior_stats['neg_pos_ratio']}")
            logger.metric(f"Number of Zero Positive Priors Ratio: {prior_stats['zero_pos_frac']}")
            logger.metric(f"Number of Min Positive Priors: {prior_stats['pos_min']}")
            logger.metric(f"Number of Mean Positive Priors: {prior_stats['pos_mean']}")
            logger.metric(f"Number of Max Positive Priors: {prior_stats['pos_max']}")
            logger.log_scalars(tag= "train", values= prior_stats, step= step)
            if isinstance(optimizer, tf.keras.mixed_precision.LossScaleOptimizer):
                logger.log_scalar(tag= "train/lr", value= optimizer.inner_optimizer.learning_rate.numpy(), step= global_step)
            else:
                logger.log_scalar(tag= "train/lr", value= optimizer.learning_rate.numpy(), step= global_step)
            
        global_step = global_step + 1
        
        # Safe guard from pushing past a certain limit
        if max_steps is not None and step + 1 >= max_steps:
            break
    
    return loss_meter.result(), global_step_offset + (step + 1)

In [40]:
amp.setup_policy()

In [43]:
train_one_epoch(config,0, model, dataset, optimizer, priors, precision_config, ema = ema, amp = amp, logger = logger)

W0000 00:00:1770236361.862895    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.867787    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.872903    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.878508    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.880322    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.882043    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.883689    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.891287    4368 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770236361.893140    4368 gp

2026-02-04 15:19:24 | ℹ️ INFO       | Epoch 0, Step 0, Loss 27.546375274658203, Num Pos: 518
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Positive Priors: 518
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Negative Priors: 1554
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Negative to Positive Ratio: 3.0
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Zero Positive Priors Ratio: 0.0
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Min Positive Priors: 126
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Mean Positive Priors: 172.6666717529297
2026-02-04 15:19:24 | ℹ️ INFO       | Number of Max Positive Priors: 243
2026-02-04 15:19:31 | ℹ️ INFO       | Epoch 0, Step 5, Loss 23.506378173828125, Num Pos: 403
2026-02-04 15:19:31 | ℹ️ INFO       | Number of Positive Priors: 403
2026-02-04 15:19:31 | ℹ️ INFO       | Number of Negative Priors: 1209
2026-02-04 15:19:31 | ℹ️ INFO       | Number of Negative to Positive Ratio: 3.0
2026-02-04 15:19:31 | ℹ️ INFO       | Number of Zero Positive 

KeyboardInterrupt: 

In [41]:
ema._num_updates

<tf.Variable 'num_updates:0' shape=() dtype=int64, numpy=0>

In [42]:
def calculate_model_prediction_health(predicted_logits: tf.Tensor, predicted_offsets: tf.Tensor, logger: Logger):
    
    # Calculating the information on the predicted logits
    probs_correct = tf.nn.softmax(predicted_logits, axis=-1)  # correct (classes)
    probs_wrong = tf.nn.softmax(predicted_logits, axis=1)   # wrong (priors)
    probabilities = tf.nn.softmax(predicted_logits, axis=-1)
    
    background_probs = probabilities[..., 0]
    foreground_probs = probabilities[..., 1:]
    
    logger.metric(f"Sum over classes (Should be 1): {tf.reduce_mean(tf.reduce_sum(probs_correct, axis=-1)).numpy()}")
    logger.metric(f"Sum over classes (should be 1 only if axis=1 softmax): {tf.reduce_mean(tf.reduce_sum(probs_wrong, axis=1)).numpy()}")
    
    logger.metric(f"Mean background probability: {tf.reduce_mean(background_probs).numpy()}")
    logger.metric(f"Max background probability: {tf.reduce_max(background_probs).numpy()}")
    
    logger.metric(f"Mean top foreground probability: {tf.reduce_mean(tf.reduce_max(foreground_probs, axis=-1)).numpy()}")
    logger.metric(f"Mean sum foreground probability: {tf.reduce_mean(tf.reduce_sum(foreground_probs, axis=-1)).numpy()}")
    logger.metric(f"Max foreground probability: {tf.reduce_max(tf.reduce_sum(foreground_probs, axis=-1)).numpy()}")
    
    logger.metric(f"Predicted Logits mean: {tf.reduce_mean(predicted_logits).numpy()}")
    logger.metric(f"Predicted Logits std: {tf.math.reduce_std(predicted_logits).numpy()}")
    logger.metric(f"Predicted Logits max: {tf.reduce_max(predicted_logits).numpy()}")
    logger.metric(f"Predicted Logits min: {tf.reduce_min(predicted_logits).numpy()}")
    

In [83]:
def evaluate_step(config: dict[str,Any],model: tf.keras.Model, priors_cxcywh: tf.Tensor, batch: dict[str, Any], precision_config: PrecisionConfig, logger: Logger):
    
    image, boxes, labels, gt_mask = batch['image'], batch['boxes'], batch['labels'], batch['gt_mask']

    tf.debugging.assert_equal(tf.rank(image), tf.constant(4, dtype = tf.int32), message = f"The image has rank : {tf.rank(image)}, expected: 4")

    # Forward pass
    predicted_offsets, predicted_logits = model(image, training = False)

    calculate_model_prediction_health(predicted_logits, predicted_offsets, logger)

    # Postprocess
    nmsed_boxes, nmsed_scores, nmsed_classes, decoded_classes, classes, valid_detections = build_decoded_boxes(config = config, predicted_offsets = predicted_offsets, predicted_logits = predicted_logits, priors = priors_cxcywh, precision_config = precision_config)

    # Decoding the GT Boxes back to image space
    image_shape = tf.shape(image)
    
    H = image_shape[1]
    W = image_shape[2]

    x1, y1, x2, y2 = tf.split(boxes, num_or_size_splits = 4, axis = -1)

    x1 = x1 * tf.cast(W, dtype = boxes.dtype)
    y1 = y1 * tf.cast(H, dtype = boxes.dtype)
    x2 = x2 * tf.cast(W, dtype = boxes.dtype)
    y2 = y2 * tf.cast(H, dtype = boxes.dtype)

    boxes = tf.concat([x1,y1,x2,y2], axis = -1)
    
    # Output the stuff in a dict
    return {
        'pred_boxes': nmsed_boxes,
        'pred_scores': nmsed_scores,
        'pred_classes': nmsed_classes,
        'gt_boxes': boxes,
        'gt_labels': labels,
        'gt_mask': gt_mask,
        'valid_detections': valid_detections,
        'class_labels': classes
    }

In [84]:
evaluate_step(config = config, model = model, priors_cxcywh = priors, batch = batch, precision_config = precision_config, logger = logger)

2026-02-09 18:55:02 | ℹ️ INFO       | Sum over classes (Should be 1): 1.0
2026-02-09 18:55:02 | ℹ️ INFO       | Sum over classes (should be 1 only if axis=1 softmax): 1.0
2026-02-09 18:55:02 | ℹ️ INFO       | Mean background probability: 0.04124898836016655
2026-02-09 18:55:02 | ℹ️ INFO       | Max background probability: 0.8198248744010925
2026-02-09 18:55:02 | ℹ️ INFO       | Mean top foreground probability: 0.23400381207466125
2026-02-09 18:55:02 | ℹ️ INFO       | Mean sum foreground probability: 0.9587510824203491
2026-02-09 18:55:02 | ℹ️ INFO       | Max foreground probability: 0.9999911189079285
2026-02-09 18:55:02 | ℹ️ INFO       | Predicted Logits mean: -0.008151971735060215
2026-02-09 18:55:02 | ℹ️ INFO       | Predicted Logits std: 1.1593793630599976
2026-02-09 18:55:02 | ℹ️ INFO       | Predicted Logits max: 7.181173324584961
2026-02-09 18:55:02 | ℹ️ INFO       | Predicted Logits min: -8.483731269836426


{'pred_boxes': <tf.Tensor: shape=(3, 100, 4), dtype=float32, numpy=
 array([[[  0.      ,   0.      , 300.      , 131.94084 ],
         [195.40236 ,  59.19565 , 300.      , 168.1453  ],
         [ 44.439888, 107.10374 , 170.40096 , 149.87189 ],
         ...,
         [249.45718 ,  84.62611 , 275.10907 , 300.      ],
         [  0.      ,  28.353237, 148.82056 , 108.53908 ],
         [ 51.134903,   0.      , 198.6948  , 134.10233 ]],
 
        [[224.1256  , 214.11084 , 300.      , 217.92494 ],
         [230.70726 ,  75.80661 , 251.85562 , 181.142   ],
         [151.2216  , 149.39398 , 198.30528 , 155.47859 ],
         ...,
         [ 21.548466, 135.64935 , 192.89027 , 226.15402 ],
         [ 10.636027,   0.      , 109.180466,  12.03889 ],
         [101.6933  ,   8.787485, 200.76291 ,  17.066605]],
 
        [[ 24.459328, 176.60571 , 129.99261 , 244.256   ],
         [ 34.694515, 300.      , 102.505974, 300.      ],
         [  0.      ,   0.      , 195.69092 , 300.      ],
         ...,

In [44]:
val_data = VOCDataset(root = config['data']['root'], split = "val", classes_file = config['data']['classes_file'], use_difficult = False)

In [45]:
val_compose = build_validation_transforms(config)

In [46]:
val_compose._transforms

In [47]:
val_dataset = create_validation_dataset(dataset = val_data, config = config, transform = val_compose)

In [48]:
batch = next(iter(val_dataset))

In [49]:
batch['boxes'] * 300

<tf.Tensor: shape=(3, 2, 4), dtype=float32, numpy=
array([[[  20.400002,    8.8     ,  268.80002 ,  234.40001 ],
        [-300.      , -300.      , -300.      , -300.      ]],

       [[  27.6     ,    9.909911,  300.      ,  300.      ],
        [  37.2     ,  171.17117 ,   49.800003,  218.91891 ]],

       [[   0.6     ,  184.      ,  256.80002 ,  234.40001 ],
        [-300.      , -300.      , -300.      , -300.      ]]],
      dtype=float32)>

In [50]:
batch['image_id']

<tf.Tensor: shape=(3,), dtype=string, numpy=array([b'2008_000002', b'2008_000003', b'2008_000007'], dtype=object)>

In [51]:
def calculate_nms_health_scores(pred_scores: tf.Tensor, valid_detections: tf.Tensor):
    
    K = tf.shape(pred_scores)[1]
    indices = tf.range(K)[tf.newaxis,:]
    
    valid_mask = indices < valid_detections[:, tf.newaxis]
    
    valid_scores = tf.boolean_mask(pred_scores, valid_mask)
    
    # Num of valid total scores
    num_valid = tf.size(valid_scores)

    # Min valid score
    min_valid = tf.cond(num_valid > 0, lambda: tf.reduce_min(valid_scores), lambda: tf.constant(0.0, dtype= pred_scores.dtype))

    # Mean valid score
    mean_valid = tf.cond(num_valid > 0, lambda: tf.reduce_mean(valid_scores), lambda: tf.constant(0.0, dtype= pred_scores.dtype))

    # Max valid score
    max_valid = tf.cond(num_valid > 0, lambda: tf.reduce_max(valid_scores), lambda: tf.constant(0.0, dtype= pred_scores.dtype))
    
    below = tf.reduce_sum(tf.cast(valid_scores < 0.9, tf.int32))

    # Average valid detections
    average_valid_detections = tf.reduce_mean(tf.cast(valid_detections, tf.float32))

    # Zero detections fractions
    zero_detections_fraction = tf.reduce_mean(tf.cast(valid_detections == 0, tf.float32))

    neg_eps = tf.constant(1e-9, dtype = pred_scores.dtype)
    masked_scores = tf.where(valid_mask, pred_scores, neg_eps)

    top1_per_image = tf.reduce_max(masked_scores, axis = 1)

    valid_top1 = tf.boolean_mask(top1_per_image, valid_detections > 0)

    # Mean Top 1 scores
    mean_top1 = tf.cond(tf.size(valid_top1) > 0, lambda: tf.reduce_mean(valid_top1), lambda: tf.constant(0.0, dtype=pred_scores.dtype))

    # Top 1 including 0 detections
    top1_incl0 = tf.where(valid_detections > 0, top1_per_image, tf.zeros_like(top1_per_image))

    # Mean Top1 including 0 detections
    mean_top1_incl0 = tf.reduce_mean(top1_incl0)

    return {
        'num_valid': num_valid,
        'min_valid': min_valid,
        'mean_valid': mean_valid,
        'max_valid': max_valid,
        'below_thresh_scores': below,
        'average_valid_det': average_valid_detections,
        'zero_valid_det': zero_detections_fraction,
        'mean_top1': mean_top1,
        'top1_incl0': top1_incl0,
        'mean_top1_incl0': mean_top1_incl0
    }

In [52]:
def calculate_gt_health_scores(ground_truth_boxes: tf.Tensor, ground_truth_labels: tf.Tensor, ground_truth_mask: tf.Tensor, select_k:int = 10):
    
    ground_truth_count = tf.reduce_sum(tf.cast(ground_truth_mask, dtype= tf.int32), axis = -1)
    avg_ground_truth_boxes_per_image = tf.reduce_mean(ground_truth_count)

    zero_ground_truth_mask = ground_truth_count == 0
    zero_ground_truth_ratio = tf.reduce_mean(tf.cast(zero_ground_truth_mask, dtype= tf.int32))

    ground_truth_per_batch = tf.boolean_mask(ground_truth_labels, ground_truth_mask)

    if tf.size(ground_truth_per_batch) != 0:
        
        unique_classes, indices, counts = tf.unique_with_counts(ground_truth_per_batch)
    
        desc_order = tf.argsort(counts, direction= 'DESCENDING')
        unique_classes = tf.gather(unique_classes, desc_order)
        counts = tf.gather(counts, desc_order)

        select_k = tf.minimum(select_k, tf.size(unique_classes))
        select_classes = tf.gather(unique_classes, tf.range(select_k))
        select_counts = tf.gather(counts, tf.range(select_k))
    else:
        select_classes = tf.constant([], dtype= tf.int32)
        select_counts = tf.constant([], dtype= tf.int32)

    return {
        'ground_truth_count': ground_truth_count,
        'avg_ground_truth_boxes_per_image': avg_ground_truth_boxes_per_image,
        'zero_ground_truth_ratio': zero_ground_truth_ratio,
        'top_gt_classes': select_classes.numpy().tolist(),
        'top_gt_class_counts': select_counts.numpy().tolist(),
        'top_gt_class_distribution': tf.repeat(select_classes, repeats= select_counts)
    }

In [53]:
def calculate_pred_health_metrics(pred_scores: tf.Tensor, pred_labels: tf.Tensor, valid_detections: tf.Tensor, select_k: int = 10, background_id: int = 0):
    
    K = tf.shape(pred_scores)[1]
    indices = tf.range(K)[tf.newaxis,:]

    valid_mask = indices < valid_detections[:, tf.newaxis]

    classes_per_batch = tf.boolean_mask(pred_labels, valid_mask)
    scores_per_batch = tf.boolean_mask(pred_scores, valid_mask)

    # Getting the foreground classes
    foreground_mask = classes_per_batch != background_id
    classes_per_batch = tf.boolean_mask(classes_per_batch, foreground_mask)
    scores_per_batch = tf.boolean_mask(scores_per_batch, foreground_mask)

    # Checking if there is something to return
    if classes_per_batch.shape.rank == 0:
        return {
        'top_classes': [],
        'top_class_countes': [],
        'top_class_distribution': tf.constant([], dtype= tf.int32)
        }

    # Saved me so much time by this implementation
    unique_classes, indices, counts = tf.unique_with_counts(classes_per_batch)

    desc_order = tf.argsort(counts, direction= 'DESCENDING')
    unique_classes = tf.gather(unique_classes, desc_order)
    counts = tf.gather(counts, desc_order)

    # Now Taking only a subset of the values for easy logging
    select_k = tf.minimum(select_k, tf.size(unique_classes))
    select_classes = tf.gather(unique_classes, tf.range(select_k))
    select_counts = tf.gather(counts, tf.range(select_k))

    return {
        'top_classes': select_classes.numpy().tolist(),
        'top_class_counts': select_counts.numpy().tolist(),
        'top_class_distribution': tf.repeat(select_classes, repeats= select_counts)
    }

In [54]:
def verify_pred_boxes_sanity(pred_boxes: tf.Tensor, valid_detections):
    K = tf.shape(pred_boxes)[1]
    indices = tf.range(K)[tf.newaxis,:]

    valid_mask = indices < valid_detections[:, tf.newaxis]
    valid_boxes = tf.boolean_mask(pred_boxes, valid_mask)

    if tf.size(valid_boxes) == 0:
        return {
            'min_coordinates': [],
            'max_coordinates': []
        }

    valid_boxes = tf.reshape(valid_boxes, [-1,4])
    min_coordinate = tf.reduce_min(valid_boxes, axis= 0)
    max_coordinate = tf.reduce_max(valid_boxes, axis= 0)

    return {
        'min_coordinates': min_coordinate.numpy().tolist(),
        'max_coordinates': max_coordinate.numpy().tolist()
    }

In [55]:
def gt_box_range(ground_truth_boxes: tf.Tensor, ground_truth_mask: tf.Tensor):
    
    ground_truth_per_batch = tf.boolean_mask(ground_truth_boxes, ground_truth_mask)

    if tf.size(ground_truth_per_batch) == 0:
        return {
            'min_coordinates': [],
            'max_coordinates': []
        }

    valid_boxes = tf.reshape(ground_truth_per_batch, [-1,4])
    min_coordinate = tf.reduce_min(valid_boxes, axis= 0)
    max_coordinate = tf.reduce_max(valid_boxes, axis= 0)

    return {
        'min_coordinates': min_coordinate.numpy().tolist(),
        'max_coordinates': max_coordinate.numpy().tolist()
    }

In [56]:
def prediction_box_bad_frac(pred_boxes: tf.Tensor, valid_detections: tf.Tensor):
    B = pred_boxes.shape[0]
    K = pred_boxes.shape[1]

    valid_mask = tf.range(K)[tf.newaxis, :] < valid_detections[:, tf.newaxis]

    x_min, y_min, x_max, y_max = tf.split(pred_boxes, 4, axis=-1)
    bad_boxes = tf.squeeze(tf.logical_or(x_max <= x_min, y_max <= y_min), axis = -1)

    valid_bad_boxes = tf.boolean_mask(bad_boxes, valid_mask)
    if tf.size(valid_bad_boxes) == 0:
        return 0.0

    return float(tf.reduce_mean(tf.cast(valid_bad_boxes, tf.float32)).numpy())

In [57]:
def ground_truth_box_bad_frac(gt_boxes: tf.Tensor, gt_mask: tf.Tensor):
    
    x_min, y_min, x_max, y_max = tf.split(gt_boxes, 4, axis=-1)
    bad_boxes = tf.squeeze(tf.logical_or(x_max <= x_min, y_max <= y_min), axis = -1)
    valid_bad_boxes = tf.boolean_mask(bad_boxes, gt_mask)

    if tf.size(valid_bad_boxes) == 0:
        return 0.0

    return float(tf.reduce_mean(tf.cast(valid_bad_boxes, tf.float32)).numpy())

In [58]:
from mobilenetv2ssd.models.ssd.ops.box_ops_tf import iou_matrix_core

def calculate_iou_sanity_top1(pred_boxes: tf.Tensor, pred_scores: tf.Tensor, valid_detections: tf.Tensor, ground_truth_boxes: tf.Tensor, gt_mask: tf.Tensor, include_no_detection_as_zero: bool = True):
    B = tf.shape(pred_boxes)[0]
    K = tf.shape(pred_boxes)[1]

    indices = tf.range(K)[tf.newaxis,:]
    valid_mask = indices < valid_detections[:, tf.newaxis]

    # For Scores
    neg_eps = tf.constant(-1e9, dtype=pred_scores.dtype)
    masked_scores = tf.where(valid_mask, pred_scores, neg_eps)
    top_index = tf.argmax(masked_scores, axis= 1, output_type= tf.int32)

    top_boxes = tf.gather(pred_boxes, top_index, batch_dims= 0)

    ious_only_det = []
    iou_incl0 = []
    num_gt_images = 0
    num_det_images = 0
    
    for index in range(int(B.numpy())):
        gt_box_image = tf.boolean_mask(ground_truth_boxes[index], gt_mask[index])
        if tf.shape(gt_box_image)[0] == 0:
            continue

        num_gt_images = num_gt_images + 1
        
        if int(valid_detections[index].numpy()) == 0:
            if include_no_detection_as_zero:
                ious_incl0.append(tf.constant(0.0, tf.float32))
            continue

        # There are detections
        num_det_images = num_det_images + 1
        top_pred = top_boxes[index]
        
        iou_matrix = iou_matrix_core(top_pred, gt_box_image)
        best_iou = tf.reduce_max(iou_matrix)

        ious_only_det.append(best_iou)
        iou_incl0.append(best_iou)
        
    mean_only_detections = tf.reduce_mean(tf.stack(ious_only_det)) if len(ious_only_det) > 0 else tf.constant(0.0, tf.float32)
    mean_incl0 = tf.reduce_mean(tf.stack(iou_incl0)) if len(iou_incl0) > 0 else tf.constant(0.0, tf.float32)
    
    # There are IoU's that need to be averaged
    return {
        "mean_iou_top1_only_det": mean_only_detections,
        "mean_iou_top1_incl0": mean_incl0,
        "num_gt_images": tf.constant(num_gt_images, tf.int32),
        "num_det_images": tf.constant(num_det_images, tf.int32),
    }

In [347]:
def evaluate(config: dict[str, Any], model: tf.keras.Model, priors_cxcywh: tf.Tensor, val_dataset: tf.data.Dataset, metrics_manager: MetricsCollection, precision_config: PrecisionConfig, ema: EMA, logger: Logger = None, max_steps: int| None = None, log_every: int = 1, heavy_log_every: int = 100):
    # Reset the metrics manager
    metrics_manager.reset()
    # Checking if the EMA is up
    # if ema.should_apply_during_eval():
    #     ema.apply_to(model)
    
    for step, batch in enumerate(val_dataset):
        # Evaluating step
        evaluation_output = evaluate_step(config = config, model = model, priors_cxcywh = priors_cxcywh, batch = batch, precision_config = precision_config, logger = logger)

        # Compute the metrics
        predictions, ground_truths = convert_batch_images_to_metric_format(pred_boxes = evaluation_output['pred_boxes'], pred_scores = evaluation_output['pred_scores'], pred_labels = evaluation_output['pred_classes'], gt_boxes = evaluation_output['gt_boxes'], gt_labels = evaluation_output['gt_labels'], gt_mask = evaluation_output['gt_mask'], image_ids = batch['image_id'])
        
        # return evaluation_output['gt_boxes'], evaluation_output['pred_boxes']
        break
        # Log the metrics
        metrics_manager.update(predictions, ground_truths)

        if max_steps is not None and step + 1 >= max_steps:
            break

        if step % heavy_log_every == 0:
            nms_health_metrics = calculate_nms_health_scores(evaluation_output['pred_scores'], evaluation_output['valid_detections'])
            ground_truth_health_metrics = calculate_gt_health_scores(batch['boxes'], batch['labels'], batch['gt_mask'])
            gt_box_sanity = gt_box_range(evaluation_output['gt_boxes'],  evaluation_output['gt_mask'])
            pred_health_metrics = calculate_pred_health_metrics(evaluation_output['pred_scores'], evaluation_output['pred_classes'], evaluation_output['valid_detections'])
            pred_box_sanity = verify_pred_boxes_sanity(evaluation_output['pred_boxes'], evaluation_output['valid_detections'])
            
            logger.metric(f"Number of Valid Detections: {evaluation_output['valid_detections']}")
            
            logger.metric(f"Mean Top1 Scores Including 0 detections:{nms_health_metrics['mean_top1_incl0']}")
            
            logger.metric(f"Average Ground Truth Count Per Image: {ground_truth_health_metrics['avg_ground_truth_boxes_per_image']}")
            logger.metric(f"Ground Truth Top Classes: {ground_truth_health_metrics['top_gt_classes']}")
            logger.metric(f"Ground Truth Top Class Counts: {ground_truth_health_metrics['top_gt_class_counts']}")
            
            logger.metric(f"Ground Truth Boxes Min Coordinate : {gt_box_sanity['min_coordinates']}")
            logger.metric(f"Ground Truth Boxes Max Coordinate : {gt_box_sanity['max_coordinates']}")
            
            logger.metric(f"Pred Top Classes : {pred_health_metrics['top_classes']}")
            logger.metric(f"Pred Top Class Counts : {pred_health_metrics['top_class_counts']}")
            
            logger.metric(f"Pred Boxes Min Coordinate : {pred_box_sanity['min_coordinates']}")
            logger.metric(f"Pred Boxes Max Coordinate : {pred_box_sanity['max_coordinates']}")
            
        if (step + 1) % log_every == 0:
            
            nms_health_metrics = calculate_nms_health_scores(evaluation_output['pred_scores'], evaluation_output['valid_detections'])
            ground_truth_health_metrics = calculate_gt_health_scores(batch['boxes'], batch['labels'], batch['gt_mask'])
            pred_health_metrics = calculate_pred_health_metrics(evaluation_output['pred_scores'], evaluation_output['pred_classes'], evaluation_output['valid_detections'])
            pred_box_sanity = verify_pred_boxes_sanity(evaluation_output['pred_boxes'], evaluation_output['valid_detections'])
            gt_box_sanity = gt_box_range(evaluation_output['gt_boxes'],  evaluation_output['gt_mask'])
            iou_sanity = calculate_iou_sanity_top1(evaluation_output['pred_boxes'], evaluation_output['pred_scores'], evaluation_output['valid_detections'], evaluation_output['gt_boxes'],  evaluation_output['gt_mask']) 
            pred_bad_boxes_ratio = prediction_box_bad_frac(evaluation_output['pred_boxes'], evaluation_output['valid_detections'])
            ground_truth_bad_box_ratio = ground_truth_box_bad_frac(evaluation_output['gt_boxes'],  evaluation_output['gt_mask'])
            
            logger.metric(f"Min Validations In Batch: {np.min(evaluation_output['valid_detections'])}")
            logger.metric(f"Mean Validations In Batch: {np.mean(evaluation_output['valid_detections'])}")
            logger.metric(f"Max Validations In Batch: {np.max(evaluation_output['valid_detections'])}")
            

            # Model NMS health metrics
            logger.metric(f"Num of Valid Scores:{nms_health_metrics['num_valid']}")
            logger.metric(f"Min Valid Scores:{nms_health_metrics['min_valid']}")
            logger.metric(f"Mean Valid Scores:{nms_health_metrics['mean_valid']}")
            logger.metric(f"Max Valid Scores:{nms_health_metrics['max_valid']}")
            logger.metric(f"Num of Valid Scores < 0.9:{nms_health_metrics['below_thresh_scores']}")
            logger.metric(f"Average Valid Detections:{nms_health_metrics['average_valid_det']}")
            logger.metric(f"Zero Valid Detection Ratio:{nms_health_metrics['zero_valid_det']}")
            logger.metric(f"Mean Top1 Scores:{nms_health_metrics['mean_top1']}")
            
            
            logger.log_scalar(tag="val/nms_num_valid_scores", value= nms_health_metrics['num_valid'], step= step)
            logger.log_scalar(tag="val/nms_min_valid_scores", value= nms_health_metrics['min_valid'], step= step)
            logger.log_scalar(tag="val/nms_mean_valid_scores", value= nms_health_metrics['mean_valid'], step= step)
            logger.log_scalar(tag="val/nms_max_valid_scores", value= nms_health_metrics['max_valid'], step= step)
            logger.log_scalar(tag="val/nms_num_valid_scores_less_than_0.9", value= nms_health_metrics['below_thresh_scores'], step= step)
            logger.log_scalar(tag="val/nms_average_valid_detections", value= nms_health_metrics['average_valid_det'], step= step)
            logger.log_scalar(tag="val/nms_zero_valid_detections_ratio", value= nms_health_metrics['zero_valid_det'], step= step)
            logger.log_scalar(tag="val/nms_mean_top1_scores", value= nms_health_metrics['mean_top1'], step= step)
            logger.log_histogram(tag="val/nms_top1_scores_incl0_detections", values= nms_health_metrics['top1_incl0'], step= step)
            logger.log_scalar(tag="val/nms_mean_top1_scores_incl0_detections", value= nms_health_metrics['mean_top1_incl0'], step= step)

            # GT Metrics
            logger.metric(f"Ground Truth Count Per Image:{ground_truth_health_metrics['ground_truth_count']}")
            logger.metric(f"Zero Ground Truth Ratio Per Batch: {ground_truth_health_metrics['zero_ground_truth_ratio']}")

            logger.metric(f"Ground Truth Boxes Bad Box Ratio : {ground_truth_bad_box_ratio}")
            logger.log_histogram(tag="val/gt_top_class_dist", values = ground_truth_health_metrics['top_gt_class_distribution'], step=step)

            logger.log_histogram(tag="val/gt_count_per_image", values= ground_truth_health_metrics['ground_truth_count'], step= step)
            logger.log_scalar(tag="val/gt_avg_count_per_image", value= ground_truth_health_metrics['avg_ground_truth_boxes_per_image'], step= step)
            logger.log_scalar(tag="val/gt_zero_ratio_per_batch", value= ground_truth_health_metrics['zero_ground_truth_ratio'], step= step)
            # logger.log_scalar(tag="val/gt_top_classes", value= ground_truth_health_metrics['top_gt_classes'], step= step)
            # logger.log_histogram(tag="val/gt_top_classes_counts", value= ground_truth_health_metrics['top_gt_class_counts'])
            # logger.log_scalar(tag="val/gt_top_classes_counts", value= ground_truth_health_metrics['top_gt_class_counts'])

            # Pred Metrics
            logger.metric(f"Pred Boxes Bad Ratio : {pred_bad_boxes_ratio}")

            logger.log_histogram(tag="val/pred_top_class_dist", values = pred_health_metrics['top_class_distribution'], step=step)
            logger.log_scalar(tag="val/pred_boxes_bad_ratio", value= pred_bad_boxes_ratio, step= step)

            # IoU Metrics
            logger.metric(f"Mean Top1 IoU Only Detection:{iou_sanity['mean_iou_top1_only_det']}")
            logger.metric(f"Mean Top1 IoU Incl0:{iou_sanity['mean_iou_top1_incl0']}")
            logger.metric(f"IoU Coverage: {iou_sanity['num_det_images']/iou_sanity['num_gt_images']}")

            # mAP Metrics
            logger.log_scalars(tag= "val",values= metrics_manager.compute(), step= step)
    
    # Restoring the EMA model variables
    inference_function(config= config, dataset_batch= batch, model_prediction= evaluation_output, logger= logger, global_step= 300)
    # ema.restore(model)
    return metrics_manager.compute()

In [418]:
def draw_bounding_boxes(image_shape: tf.Tensor, image_id: tf.Tensor, boxes: tf.Tensor, labels: tf.Tensor, pred_boxes: tf.Tensor, pred_scores: tf.Tensor, pred_labels: tf.Tensor, dataset_name: str, dataset_root: str, labels_map: dict[str, int]| None = None):
    from PIL import ImageDraw
    from pathlib import Path
    
    if dataset_name == "voc":
        dataset_root = Path(dataset_root)
        dataset_root = dataset_root / "JPEGImages"
        image_file = dataset_root / f"{image_id.numpy().decode()}.jpg"
    else:
        raise ValueError("Wrong Dataset Type")

    original_image = Image.open(image_file)
    H, W = image_shape[0], image_shape[1]
    original_image = original_image.resize((W,H))
    draw = ImageDraw.Draw(original_image)

    def label_color(l):
        return ((37 * l + 17) % 256, (57 * l + 101) % 256, (83 * l + 59) % 256)

    # Draw ground truth boxes
    for i in range(boxes.shape[0]):
        x1, y1, x2, y2 = boxes[i].numpy()
        c = label_color(int(labels[i].numpy()))
        draw.rectangle([x1, y1, x2, y2], outline=c, width=2)
        draw.text((x1, y1 - 10), f"GT:{labels_map[int(labels[i])]}", fill=c)

    # Draw prediction boxes
    for i in range(pred_boxes.shape[0]):
        x1, y1, x2, y2 = pred_boxes[i].numpy()
        c = label_color(int(pred_labels[i].numpy()))
        score = float(pred_scores[i].numpy())
        draw.rectangle([x1, y1, x2, y2], outline=c, width=2)
        draw.text((x1, y2 + 2), f"P:{labels_map[int(pred_labels[i])]} {score:.2f}", fill=c)

    y_offset = 10
    unique_labels = set(labels.numpy().tolist()) | set(pred_labels.numpy().tolist())
    for lid in unique_labels:
        c = label_color(int(lid))
        draw.rectangle([5, y_offset, 20, y_offset + 12], fill=c)
        draw.text((25, y_offset), labels_map[int(lid)], fill=c)
        y_offset += 16

    result = tf.constant(np.array(original_image), dtype=tf.float32)
    return result / 255.0

In [421]:
def inference_function(config: dict[str,Any], dataset_batch: dict[str, Any], model_prediction: dict[str, Any], logger: Logger, global_step: int, top_k_per_image: int = 5):
    # Taking the first image from the batch
    gather_index= tf.constant([2], dtype=tf.int32)
    image= tf.gather(dataset_batch['image'], gather_index)
    image_id= tf.gather(dataset_batch['image_id'], gather_index)
    gt_boxes = tf.gather(model_prediction['gt_boxes'], gather_index)
    gt_mask = tf.gather(dataset_batch['gt_mask'], gather_index)
    gt_labels = tf.gather(dataset_batch['labels'], gather_index)

    image= tf.squeeze(image, axis= 0)
    image_id= tf.squeeze(image_id)
    valid_gt= tf.boolean_mask(gt_boxes, gt_mask)
    valid_gt_labels= tf.boolean_mask(gt_labels, gt_mask)
    
    pred_labels= tf.gather(model_prediction['pred_classes'], gather_index)
    pred_labels= tf.squeeze(pred_labels, axis= 0)
    pred_scores = tf.gather(model_prediction['pred_scores'], gather_index)
    pred_scores= tf.squeeze(pred_scores, axis= 0)
    pred_boxes = tf.gather(model_prediction['pred_boxes'], gather_index)
    pred_boxes= tf.squeeze(pred_boxes, axis= 0)
    
    # Choosing the labels
    top_k_scores, top_k_indices = tf.math.top_k(pred_scores, k= top_k_per_image, sorted=True)
    top_k_boxes= tf.gather(pred_boxes, top_k_indices)
    top_k_labels= tf.gather(pred_labels, top_k_indices)

    img= draw_bounding_boxes(image_shape= tf.shape(image), image_id= image_id, boxes= valid_gt,labels= valid_gt_labels,pred_boxes= top_k_boxes, pred_scores= top_k_scores, pred_labels= top_k_labels, dataset_name = config['data']['dataset_name'],dataset_root = config['data']['root'],labels_map= model_prediction['class_labels'])
    
    logger.success(f"Logged eval image....{'.'*20}")

In [422]:
evaluate(config, model, priors, val_dataset, metrics_manager = metrics_manager, precision_config = precision_config, logger = logger,ema = ema)

2026-02-09 23:12:26 | ℹ️ INFO       | Sum over classes (Should be 1): 1.0
2026-02-09 23:12:26 | ℹ️ INFO       | Sum over classes (should be 1 only if axis=1 softmax): 1.0
2026-02-09 23:12:26 | ℹ️ INFO       | Mean background probability: 0.04124898836016655
2026-02-09 23:12:26 | ℹ️ INFO       | Max background probability: 0.8198248744010925
2026-02-09 23:12:26 | ℹ️ INFO       | Mean top foreground probability: 0.23400381207466125
2026-02-09 23:12:26 | ℹ️ INFO       | Mean sum foreground probability: 0.9587510824203491
2026-02-09 23:12:26 | ℹ️ INFO       | Max foreground probability: 0.9999911189079285
2026-02-09 23:12:26 | ℹ️ INFO       | Predicted Logits mean: -0.008151971735060215
2026-02-09 23:12:26 | ℹ️ INFO       | Predicted Logits std: 1.1593793630599976
2026-02-09 23:12:26 | ℹ️ INFO       | Predicted Logits max: 7.181173324584961
2026-02-09 23:12:26 | ℹ️ INFO       | Predicted Logits min: -8.483731269836426
2026-02-09 23:12:26 | ℹ️ INFO       | Logged eval image.................

{'voc_ap_50/mAP@0.50': 0.0,
 'voc_ap_50/mAP@0.75': 0.0,
 'coco_map/mAP@0.50': 0.0,
 'coco_map/mAP@0.75': 0.0}

In [61]:
# evaluate_step(config, model, priors, val_info, precision_config, logger)

In [62]:
def fit(config: dict[str,Any], model: tf.keras.Model, priors_cxcywh: tf.Tensor, train_dataset: tf.data.Dataset, validation_dataset: tf.data.Dataset, optimizer: tf.keras.optimizers.Optimizer, precision_config: PrecisionConfig, metrics_manager: MetricsCollection, logger: Logger, checkpoint_manager: CheckpointManager, ema: EMA, amp: AMPContext, start_epoch: int = 0, global_step: int = 0, max_epochs: int | None = None, best_metric: float | None = None):
    # Initialize overarching variables
    # 1. Epoch, 2. eval_every, 3. log_every, 4. heavy_log_every, 5. save_every, 6. save_best, 7. best_metric, 8. global_step
    epochs = max_epochs if max_epochs is not None else int(config['train']['epochs'])
    eval_every = int(config['train'].get('eval_every', 1))
    train_log_every = int(config['logging'].get('log_interval_steps', 10))
    eval_log_every = int(config['logging'].get('log_interval_steps', 10))

    if best_metric is None:
        best_metric = float("-inf")

    primary_metric = config['eval'].get('main_metric', 'voc_ap_50')

    logger.metric(f"Starting fit: epochs={epochs}, start_epoch={start_epoch}, global_step={global_step}")
    
    # Loop over the epochs:
        # Train one epoch
        # Log Training loss, learning_rate_at_epoch_end
        # Check if evaluate necessary:
            # Evaluate model
            # Log Metrics
            # Decide if to save best model
                # Save best
        # Save checkpoint at the end (model weights, optimizer state, epoch, global step, best_metric, EMA weights)

    for epoch in range(start_epoch, epochs):
        logger.metric(f"Epoch {epoch+1}/{epochs} starting")

        # Training over one epoch
        train_loss, global_step = train_one_epoch(config, epoch, model, train_dataset, optimizer, priors_cxcywh, precision_config, ema = ema, amp = amp, logger = logger)

        # Logging the scalar
        logger.log_scalar("train/loss_epoch_mean", float(train_loss.numpy()), step=global_step)

        if epoch % eval_every == 0 or epoch == epochs - 1:
            # Evaluate the model
            eval_metrics = evaluate(config, model, priors_cxcywh, validation_dataset, metrics_manager = metrics_manager, precision_config = precision_config, logger = logger,ema = ema)
            logger.log_scalars(tag = "val", values= eval_metrics, step= global_step)

            # Checking for the best metric
            score = float(eval_metrics.get(primary_metric, float("-inf")))
            if score > best_metric:
                best_metric = score
                logger.metric(f"New Best {primary_metric}: {best_metric}")

                if checkpoint_manager is not None:
                    checkpoint_manager.save_best(epoch= epoch, global_step= global_step, metric= best_metric)

        # Checkpointing the last model at the end of the epoch
        if checkpoint_manager is not None:
            checkpoint_manager.save_last(epoch= epoch, global_step= global_step)

        # Logging the end of the model
        logger.metric(f"Epoch {epoch + 1} done. best_{primary_metric}={best_metric}")

    # Return Training Summary [final_epoch_metrics, best_metric, checkpoint_path]
    return {
        'best_metric': best_metric,
        'primary_metric': primary_metric,
        'global_step': global_step
    }   

In [63]:
fit(config, model, priors,train_dataset = dataset, validation_dataset= val_dataset,optimizer= optimizer,precision_config = precision_config, metrics_manager= metrics_manager, logger= logger, checkpoint_manager= checkpoint_manager, ema = ema, amp= amp) 

2026-02-09 15:53:20 | ℹ️ INFO       | Starting fit: epochs=50, start_epoch=0, global_step=0
2026-02-09 15:53:20 | ℹ️ INFO       | Epoch 1/50 starting


W0000 00:00:1770670398.964336     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.970385     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.973435     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.982576     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.984780     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.986913     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.989001     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670398.999929     833 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770670399.002233     833 gp

2026-02-09 15:53:23 | ℹ️ INFO       | Epoch 0, Step 0, Loss 27.59795379638672, Num Pos: 518
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Positive Priors: 518
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Negative Priors: 1554
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Negative to Positive Ratio: 3.0
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Zero Positive Priors Ratio: 0.0
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Min Positive Priors: 126
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Mean Positive Priors: 172.6666717529297
2026-02-09 15:53:23 | ℹ️ INFO       | Number of Max Positive Priors: 243
2026-02-09 15:53:38 | ℹ️ INFO       | Epoch 0, Step 5, Loss 23.54554557800293, Num Pos: 403
2026-02-09 15:53:38 | ℹ️ INFO       | Number of Positive Priors: 403
2026-02-09 15:53:38 | ℹ️ INFO       | Number of Negative Priors: 1209
2026-02-09 15:53:38 | ℹ️ INFO       | Number of Negative to Positive Ratio: 3.0
2026-02-09 15:53:38 | ℹ️ INFO       | Number of Zero Positive Pr

KeyboardInterrupt: 

In [68]:
checkpoint_manager.save_last(epoch= 0, global_step= 200)

'/mnt/d/dev/MobileNetV2-SSD/checkpoints/mobilenetv2-ssd_voc_img224_bs3_lr1.00e-03_cosine_warmup_465f0dfe5d/last/ckpt-200'